# War driving; protecting yourself (verify SSIDs, avoid open networks, VPN)

# Coding problem, 1.3.d: Public Wi-Fi risk flagger

**Skill:** Mitigate Risk · **Maps to:** 1.3.B, 1.3.C · **Language:** Python 3 (pytest)

## Problem
You are building the risk-assessment routine a network-policy tool runs before letting a device
connect to a public Wi-Fi network. Given a description of one network and the user's current
posture, return the **risk flags** that apply, so the user (or an administrator) can make an
informed decision. This is exactly the three-step personal protocol from the lesson, expressed
as code: verify the SSID, avoid open networks, always use a VPN.

The function is pure and deterministic, no network, no I/O.

## Function contract
Implement `solve(network: dict) -> list[str]` in the code editor (see the Reference section below).

## Input
`network`, a `dict` describing one network and the user's posture toward it. Recognized keys:

| Key | Type | Meaning |
|-----|------|---------|
| `"encryption"` | `str` | Wi-Fi security type, e.g. `"open"`, `"wpa2"`, `"wpa3"`. Case-insensitive. Missing / `None` / `""` is treated as open (worst case). |
| `"verified"` | `bool` | `True` if the SSID was confirmed with venue staff / an official source. Missing is treated as `False`. |
| `"vpn"` | `bool` | `True` if a VPN tunnel is active on the device. Missing is treated as `False`. |

Example:
```python
{"ssid": "AirportFreeWifi", "encryption": "open", "verified": False, "vpn": False}
```

## Output
A **sorted, de-duplicated** `list[str]` of the risk flags that apply, drawn from:

- `"open-network"`, the network is open (no Wi-Fi-layer encryption).
- `"unverified-ssid"`, the SSID has not been confirmed with an authoritative source.
- `"no-vpn"`, no VPN is protecting the user's traffic.

Return `[]` when none apply (a verified, encrypted network used over a VPN).

## Examples
```
{"encryption": "open", "verified": False, "vpn": False}   -> ["no-vpn", "open-network", "unverified-ssid"]
{"encryption": "wpa2", "verified": True,  "vpn": True}     -> []
{"encryption": "wpa2", "verified": True,  "vpn": False}    -> ["no-vpn"]
{"encryption": "WPA3", "verified": False, "vpn": True}     -> ["unverified-ssid"]
{}                                                         -> ["no-vpn", "open-network", "unverified-ssid"]
```

## Constraints
- Deterministic, offline, standard library only.
- Matching of `encryption` is case-insensitive.
- Output always sorted ascending and de-duplicated.
- Must pass all cases in the tests (visible + hidden).

## Reference
Any implementation that satisfies this interface can be swapped in (reference, student, or
composite) and must pass the tests unchanged.

### Entry point
```python
def solve(network: dict) -> list[str]
```

### Input (`network`)
A `dict` describing one Wi-Fi network and the user's posture toward it. Recognized keys
(any may be absent; extra keys such as `"ssid"` are ignored):

- `"encryption"`: `str`, Wi-Fi security type (`"open"`, `"wpa2"`, `"wpa3"`...). Compared
  case-insensitively. Missing, `None`, or `""` counts as **open**.
- `"verified"`: `bool`, `True` if the SSID was confirmed with an authoritative source.
  Missing counts as `False`.
- `"vpn"`: `bool`, `True` if a VPN tunnel is active. Missing counts as `False`.

### Output (`result`)
A **sorted, de-duplicated** `list[str]` drawn from
`{"no-vpn", "open-network", "unverified-ssid"}`. Empty list if no risk applies.

### Risk rules
- **open-network**, `encryption`, lowercased, is one of `{"open", "none", ""}` (or the key is
  absent / `None`). Encrypted types (`wpa`, `wpa2`, `wpa3`, anything else non-empty) do not flag.
- **unverified-ssid**, `verified` is not truthy.
- **no-vpn**, `vpn` is not truthy.

### Guarantees / rules
- Pure function, deterministic, no I/O, no network. Standard library only.
- `encryption` matching is case-insensitive.
- Output always sorted ascending and de-duplicated.

### Acceptance
Your code passes when every test below shows a checkmark (some are hidden until you run them).


In [ ]:
# Your implementation
"""Starter code for 1.3.d. Student completes the TODO.
This file intentionally FAILS the tests until implemented."""
from typing import List


def solve(network: dict) -> List[str]:
    """Return the sorted, de-duplicated risk flags for one network description.

    Flags: "open-network", "unverified-ssid", "no-vpn".
    See the problem statement above and the Reference section below for the rules.

    Args:
        network: dict with optional keys "encryption" (str), "verified" (bool), "vpn" (bool).
    Returns:
        sorted list of the risk-flag strings that apply ([] if none).
    """
    # TODO: flag "open-network" when encryption is open/missing,
    #       "unverified-ssid" when not verified, "no-vpn" when no VPN active.
    #       Return the flags sorted and de-duplicated.
    raise NotImplementedError("implement solve() — see the problem statement above")


## Self-check tests
Run the cell below to check your `solve` implementation above.

In [ ]:
import __main__ as impl  # the functions defined above

# ---- visible cases -------------------------------------------------------
def test_worst_case_open_unverified_no_vpn():
    out = impl.solve({"ssid": "AirportFreeWifi", "encryption": "open",
                      "verified": False, "vpn": False})
    assert out == ["no-vpn", "open-network", "unverified-ssid"]


def test_fully_protected():
    assert impl.solve({"encryption": "wpa2", "verified": True, "vpn": True}) == []


def test_only_missing_vpn():
    assert impl.solve({"encryption": "wpa2", "verified": True, "vpn": False}) == ["no-vpn"]


def test_encrypted_but_unverified_with_vpn():
    assert impl.solve({"encryption": "WPA3", "verified": False, "vpn": True}) == ["unverified-ssid"]


# ---- hidden / edge cases -------------------------------------------------
def test_empty_dict_is_worst_case():
    # Missing encryption -> open; missing verified -> False; missing vpn -> False.
    assert impl.solve({}) == ["no-vpn", "open-network", "unverified-ssid"]


def test_encryption_case_insensitive():
    assert impl.solve({"encryption": "OpEn", "verified": True, "vpn": True}) == ["open-network"]


def test_encryption_none_and_empty_count_as_open():
    assert impl.solve({"encryption": None, "verified": True, "vpn": True}) == ["open-network"]
    assert impl.solve({"encryption": "", "verified": True, "vpn": True}) == ["open-network"]
    assert impl.solve({"encryption": "none", "verified": True, "vpn": True}) == ["open-network"]


def test_wep_is_not_open_network():
    # WEP is weak, but it is encrypted -> not the "open-network" flag (only verify/vpn matter here).
    assert impl.solve({"encryption": "wep", "verified": True, "vpn": True}) == []


def test_output_sorted_and_deduplicated():
    out = impl.solve({"encryption": "open"})  # everything missing -> all three flags
    assert out == ["no-vpn", "open-network", "unverified-ssid"]
    assert out == sorted(set(out))


def test_extra_keys_ignored():
    out = impl.solve({"encryption": "wpa2", "verified": True, "vpn": True,
                      "ssid": "Cafe", "signal": -42, "bssid": "aa:bb:cc"})
    assert out == []


# ---- run every test above ------------------------------------------------
_results = []
for _name, _fn in list(globals().items()):
    if _name.startswith("test_") and callable(_fn):
        try:
            _fn()
            _results.append((_name, None))
        except Exception as _e:
            _results.append((_name, _e))
for _name, _e in _results:
    print(("PASS  " if _e is None else "FAIL  ") + _name + ("" if _e is None else f": {_e!r}"))
print(f"{sum(e is None for _, e in _results)}/{len(_results)} tests passed")
